In [0]:
%pip install databricks-ai-search
dbutils.library.restartPython()

In [0]:
import re
from pyspark.sql import functions as F
from databricks.ai_search.client import AISearchClient

catalog, schema = "workspace", "patient_notes"
chunk_table   = f"{catalog}.{schema}.silver_chunks"
summary_table = f"{catalog}.{schema}.gold_patient_summaries"
llm           = "databricks-meta-llama-3-3-70b-instruct"

index = AISearchClient(disable_notice=True).get_index(index_name=f"{catalog}.{schema}.silver_chunks_index")

USEFUL_SECTIONS = ["chief complaint", "history of present illness", "hospital course", "assessment"]   # no "plan"
MIN_SCORE = 0.45

In [0]:
def call_llm(prompt):
    df = spark.createDataFrame([(prompt,)], "prompt string")        # a 1-row DataFrame avoids quoting problems
    return df.select(F.expr(f"ai_query('{llm}', prompt)").alias("answer")).first()["answer"]

In [0]:
def retrieve(patient_id, question, k=8):
    res = index.similarity_search(
        query_text=question,
        columns=["chunk_id", "note_id", "note_date", "section", "chunk_text"],
        filters={"patient_id": patient_id, "section": USEFUL_SECTIONS},   # this patient, useful sections only
        num_results=k)
    rows = res["result"]["data_array"]                      # each row: columns..., then the score last
    kept, seen = [], set()
    for r in rows:
        body = r[4].split(" · ")[-1]                        # text after the contextual header
        if r[-1] >= MIN_SCORE and body not in seen:         # score threshold + de-duplication
            seen.add(body)
            kept.append(r)
    return kept

In [0]:
def ask_patient(patient_id, question):
    chunks = retrieve(patient_id, question)
    if not chunks:
        return "Not found in this patient's notes.", []
    context = "\n".join(f"[{r[1]}] {r[4]}" for r in chunks)
    prompt = f"""You are a clinical assistant. Answer using ONLY the note excerpts below.
Put the note id in square brackets after each fact, for example [N20261101-002].
If the excerpts do not answer the question, reply exactly: Not found in this patient's notes.
Do not guess and do not add outside knowledge.

Question: {question}

Note excerpts:
{context}

Answer:"""
    return call_llm(prompt), chunks

answer, used = ask_patient("P0012", "Has this patient had any surgery?")
print(answer)
for r in used:
    print(r[1], r[3], round(r[-1], 3))

In [0]:
def summarize_patient(patient_id):
    rows = (spark.table(chunk_table)
            .filter((F.col("patient_id") == patient_id) &
                    F.col("section").isin("history of present illness", "assessment"))
            .orderBy("note_date", "note_id")
            .collect())
    note_ids = sorted({r.note_id for r in rows})
    context  = "\n".join(f"[{r.note_id}] {r.chunk_text}" for r in rows)
    prompt = f"""You are writing a patient summary for a doctor. Use ONLY the note excerpts below.
Write at most 6 short bullet points under these headings:
Active now / Pre-existing / Past history and surgeries / Ruled out.
Put the note id in square brackets after every fact, for example [N20261101-002].
If a heading has nothing, write: None documented.
Do not guess and do not add outside knowledge.

Note excerpts:
{context}

Summary:"""
    return call_llm(prompt), note_ids

In [0]:
def check_citations(text, allowed_ids):
    cited = set(re.findall(r"N\d{8}-\d{3}", text))
    return {"cited": len(cited), "invalid": sorted(cited - set(allowed_ids))}

In [0]:
patients_to_run = ["P0012", "P0026", "P0022"]   # replace the last one

rows = []
for pid in patients_to_run:
    summary, ids = summarize_patient(pid)
    chk = check_citations(summary, ids)
    rows.append((pid, summary, len(ids), chk["cited"], len(chk["invalid"])))
    print(pid, "| notes used:", len(ids), "| citations:", chk["cited"], "| invalid:", chk["invalid"])
    print(summary)
    print("-" * 70)

(spark.createDataFrame(rows, "patient_id string, summary string, notes_used int, citations int, invalid_citations int")
      .withColumn("created_at", F.current_timestamp())
      .write.mode("append").saveAsTable(summary_table))